# RAG Financial Research Assistant with Source Attribution

**Domain:** AI Engineering / LLMs / Financial Research  
**Primary Evaluation Focus:** Retrieval Quality, Source Coverage, Answer Traceability  
**Dataset:** Financial document corpus

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- RAG systems reduce hallucination risk by grounding answers in retrieved documents.
- Source attribution is essential for financial research trust and auditability.
- Chunking, retrieval strategy and evaluation are more important than the LLM alone.
- A production system should include citations, confidence signals and fallback behaviour.
- The project demonstrates applied AI engineering beyond simple prompt-based usage.

## Business Recommendations

- Use vector search with metadata filters for production retrieval.
- Evaluate retrieval using known-answer test questions.
- Return sources with every generated answer.
- Add guardrails for unsupported or low-confidence responses.
- Deploy behind an API with logging, monitoring and access controls.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **Retrieval Quality, Source Coverage, Answer Traceability**.

# Project 09 - RAG Financial Research Assistant
**Domain:** AI / LLM / AI Engineering

Production-grade Retrieval-Augmented Generation for financial document Q&A with source citation.

`pip install langchain langchain-community openai faiss-cpu fastapi uvicorn`

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')

# Optional: set your OpenAI API key
# os.environ['OPENAI_API_KEY'] = 'sk-...'

try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    LANGCHAIN = True
    print("LangChain available")
except:
    LANGCHAIN = False
    print("LangChain not installed - using demo mode (keyword retrieval)")
    print("pip install langchain langchain-community openai faiss-cpu")

In [ ]:
# ── FINANCIAL DOCUMENT CORPUS ─────────────────────────────────
documents = [
    {
        "source": "LLOY_2023_Annual_Report.pdf",
        "text": (
            "Lloyds Banking Group delivered statutory profit before tax of 7.5 billion in 2023, "
            "a 57 percent increase on prior year. Net interest income grew to 13.8 billion. "
            "Net interest margin improved to 3.11 percent. CET1 ratio of 14.1 percent. "
            "Customer deposits grew 2 percent to 475 billion. Return on tangible equity 15.8 percent. "
            "Cost-to-income ratio improved to 49.4 percent. Final dividend of 1.84 pence per share."
        )
    },
    {
        "source": "BARC_2023_Annual_Report.pdf",
        "text": (
            "Barclays reported return on tangible equity of 10.6 percent for 2023. "
            "Total income was 25.4 billion. Investment banking contributed 10.2 billion. "
            "Credit impairment charges increased to 1.9 billion. CET1 ratio of 13.8 percent. "
            "Targeting 1 billion of structural cost reduction by 2026."
        )
    },
    {
        "source": "NWG_2023_Annual_Report.pdf",
        "text": (
            "NatWest Group reported attributable profit of 4.0 billion. "
            "Return on tangible equity of 17.8 percent. Net interest margin expanded to 3.13 percent. "
            "Total income increased 14 percent to 14.8 billion. "
            "UK government reduced its stake to below 30 percent. "
            "Cost-to-income ratio 48 percent. Returned 4.6 billion to shareholders."
        )
    },
    {
        "source": "SECTOR_OUTLOOK_2024.pdf",
        "text": (
            "UK banking sector outlook 2024: Net interest margins expected to compress 20-40bps as base rate cuts begin. "
            "Mortgage volumes forecast to recover as affordability improves. "
            "Credit quality remains resilient with unemployment stable. "
            "ESG regulatory requirements will increase compliance costs. "
            "Digital challenger banks continue to gain market share. "
            "M&A activity likely to accelerate as valuations remain compressed."
        )
    },
]
print(f"Loaded {len(documents)} documents")
for d in documents:
    print(f"  {d['source']}: {len(d['text'].split())} words")

In [ ]:
# ── RAG RETRIEVAL ENGINE ──────────────────────────────────────
class KeywordRetriever:
    def __init__(self, docs):
        self.docs = docs

    def retrieve(self, query, k=2):
        query_words = set(query.lower().split())
        scored = []
        for doc in self.docs:
            words = set(doc['text'].lower().split())
            score = len(query_words & words)
            scored.append((score, doc))
        scored.sort(reverse=True)
        return [d for _, d in scored[:k]]

retriever = KeywordRetriever(documents)

def rag_query(query, k=2):
    retrieved = retriever.retrieve(query, k=k)
    print(f"Query: {query}")
    print(f"Sources used: {[d['source'] for d in retrieved]}")

    # Extract most relevant sentences
    query_words = set(query.lower().split())
    answers = []
    for doc in retrieved:
        for sent in doc['text'].split('.'):
            if any(w in sent.lower() for w in query_words) and len(sent.strip()) > 20:
                answers.append(f"{sent.strip()}. [Source: {doc['source']}]")

    print("Answer:")
    for a in answers[:3]:
        print(f"  {a}")
    print()

# Test queries
rag_query("What was Lloyds profit in 2023?")
rag_query("Which bank has the best CET1 ratio?")
rag_query("What are UK banking risks for 2024?")
rag_query("NatWest return on equity performance")

In [ ]:
# ── PRODUCTION FAISS + OPENAI SETUP ──────────────────────────
print("""
=== Production RAG Setup ===

1. Install dependencies:
   pip install langchain langchain-community openai faiss-cpu tiktoken

2. Set your OpenAI key:
   import os
   os.environ['OPENAI_API_KEY'] = 'sk-...'

3. Build FAISS vector store:
   from langchain_community.vectorstores import FAISS
   from langchain_community.embeddings import OpenAIEmbeddings

   texts = [doc['text'] for doc in documents]
   embeddings = OpenAIEmbeddings()
   vectorstore = FAISS.from_texts(texts, embeddings)

4. Run a query:
   from langchain.chains import RetrievalQA
   from langchain_community.llms import OpenAI

   qa = RetrievalQA.from_chain_type(
       llm=OpenAI(temperature=0),
       retriever=vectorstore.as_retriever(search_kwargs={'k':3})
   )
   result = qa.run('What was Lloyds profit in 2023?')
   print(result)
""")

In [ ]:
# ── FASTAPI SERVICE BLUEPRINT ─────────────────────────────────
blueprint = """
# main.py — FastAPI RAG Service

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title='RAG Financial Research Assistant', version='1.0')

class Query(BaseModel):
    question: str
    k: int = 3

class Answer(BaseModel):
    answer: str
    sources: list
    confidence: float

@app.post('/query', response_model=Answer)
async def query(q: Query):
    retrieved = retriever.retrieve(q.question, k=q.k)
    # In production: pass context to GPT-4 / Claude
    context = ' '.join([d['text'] for d in retrieved])
    answer = f'Based on {len(retrieved)} documents: ' + context[:200] + '...'
    return Answer(
        answer=answer,
        sources=[d['source'] for d in retrieved],
        confidence=0.91
    )

# Run: uvicorn main:app --host 0.0.0.0 --port 8000
# Test: curl -X POST http://localhost:8000/query -H 'Content-Type: application/json' -d '{"question":"Lloyds profit?"}'
"""

print(blueprint)

# Final Business Interpretation

## Key Findings

- RAG systems reduce hallucination risk by grounding answers in retrieved documents.
- Source attribution is essential for financial research trust and auditability.
- Chunking, retrieval strategy and evaluation are more important than the LLM alone.
- A production system should include citations, confidence signals and fallback behaviour.
- The project demonstrates applied AI engineering beyond simple prompt-based usage.

## Recommended Next Steps

- Use vector search with metadata filters for production retrieval.
- Evaluate retrieval using known-answer test questions.
- Return sources with every generated answer.
- Add guardrails for unsupported or low-confidence responses.
- Deploy behind an API with logging, monitoring and access controls.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Designed a retrieval-augmented financial research assistant with source attribution, retrieval logic and a FastAPI deployment blueprint for auditable document Q&A."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining